# AutoML: automated model comparison

By now you've built classifiers by hand — [logistic
regression](../02-regression/logistic-regression.ipynb), [decision
trees](../04-trees/decision-trees.ipynb) — and learned to score them with
[cross-validation](../01d-evaluation/cross-validation.ipynb). **AutoML**
automates the tedious part: it trains a whole *zoo* of models on your data,
cross-validates each, and hands you a ranked leaderboard.

```{note}
"AutoML" here means **automated model selection + cross-validation over a fixed
set of algorithms** — not neural architecture search. It's running the exact CV
loop from the evaluation chapter, once per model, so you don't have to.
```

We use the [`automl`](https://github.com/cmccomb/rust-automl) crate (built on
`smartcore`). It's distributed via git, not crates.io.

## Compare models with one call

Give it a feature matrix and unsigned integer labels, pick
`ClassificationSettings::default()`, and call `.train()`. Printing the model
renders the leaderboard.

In [ ]:
:dep automl = { git = "https://github.com/cmccomb/rust-automl" }
use automl::ClassificationModel;
use automl::settings::ClassificationSettings;
use automl::DenseMatrix;  // automl re-exports smartcore's matrix type

// Two roughly-separated classes of 2-D points (labels must be unsigned ints).
let x = DenseMatrix::from_2d_array(&[
    &[1.0, 2.0], &[2.0, 1.0], &[1.5, 1.8], &[1.2, 2.2], &[0.8, 1.5],
    &[1.9, 1.1], &[1.3, 1.7], &[0.9, 2.1], &[2.5, 2.6], &[3.0, 2.0],
    &[8.0, 9.0], &[9.0, 8.0], &[8.5, 8.8], &[9.2, 9.1], &[7.8, 8.2],
    &[8.3, 9.3], &[9.1, 7.9], &[8.7, 8.5], &[6.8, 7.2], &[7.5, 6.6],
]).unwrap();
let y: Vec<u32> = vec![0,0,0,0,0,0,0,0,0,0, 1,1,1,1,1,1,1,1,1,1];

let mut model = ClassificationModel::new(x, y, ClassificationSettings::default());
model.train().unwrap();
println!("{model}");

## Reading the leaderboard

Each row is a model, with its cross-validated **training** and **testing**
score and how long it took to fit. The table is sorted best-first, so the top
row is `automl`'s pick. Compare *testing* score (generalization), not training
score — a model that's perfect on training but worse on testing is overfitting,
exactly the lesson from the [evaluation chapter](../01d-evaluation/cross-validation.ipynb).

On cleanly separable data like this, several models score identically — the
*timings* then become the tie-breaker. On messy real data (e.g. the
[ETL chapter's](../01c-etl/data-preparation.ipynb) customer dataset) the scores
spread out and the comparison earns its keep.

## Preprocessing pipelines

`automl` can also run feature preprocessing before training via its
`PreprocessingPipeline` / `PreprocessingStep` API (imputation, scaling), attached
with `ClassificationSettings::default().with_preprocessing(...)`. That's the
automated version of what you did by hand in the
[ETL chapter](../01c-etl/data-preparation.ipynb) — see `automl`'s `cookbook`
module and examples for the current API, which is evolving.

```{warning}
**Set expectations.** `automl` is a smaller-scope tool than Python's
`auto-sklearn` or `TPOT`. It compares a **fixed model zoo** via cross-validation;
it does **not** do full hyperparameter search or neural architecture search, and
it's still under active development (hence the git dependency). Treat it as "a
fast way to compare the usual models," not a drop-in scikit-learn AutoML
equivalent.
```

To go further with the winning model's *own* hyperparameters, combine this with
the [hyperparameter search](../05b-optimization/hyperparameter-search.ipynb)
chapter. Next: [Explainable ML](../07-explainability/model-interpretability.ipynb) —
understanding *why* the chosen model predicts what it does.